# Авито: классификатор «запрос → микрокатегория» (признак для CatBoost)

**Входы:** результат предобработки (`pairs.parquet`, `queries.parquet`, `items.parquet`) и результат ноутбука признаков (`features.json`, `train.parquet`, `val.parquet`). Файлы ищутся в `/kaggle/input`, потом в `/kaggle/working`. GPU не нужен.

Что делает ноутбук:
1. Учит линейный классификатор предсказывать по тексту запроса и его фильтру микрокатегорию выбранного объявления.
   Учим на **всех train-запросах**, включая те ~90%, что не попали в CatBoost
2. Предсказывает топ-30 микрокатегорий с вероятностями для запросов CatBoost-train, val и бенчмарка
3. Сохраняет `microcat_proba.parquet` (`qid, microcat, proba`). CatBoost-ноутбук присоединит его к кандидатам по `(qid, item_microcat_id)`

**Защита от утечки.** Если классификатор видел запрос при обучении, на этом запросе он будет уверен неестественно сильно.
Для 15 тыс. запросов CatBoost-train делаем **out-of-fold** предсказания: делим их тексты на 3 фолда, и каждый фолд
предсказывается моделью, которая не видела ни одного запроса с такими текстами. Так условия совпадают с val, где тексты
тоже невиданные. Val и бенчмарк предсказываются финальной моделью, обученной на всех train-запросах.

### Импорты, пути, параметры

In [ ]:
import os, re, gc, glob, time, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

def find_dir(filename, roots=('/kaggle/input', '/kaggle/working', '.')):
    """Папка, где лежит filename: сначала подключённые входы (/kaggle/input), потом рабочая папка."""
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', filename), recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError(f'{filename} не найден ни в /kaggle/input, ни в /kaggle/working')

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'

PREP_DIR = find_dir('pairs.parquet')           # предобработка
FEAT_DIR = find_dir('features.json')           # признаки CatBoost (нужны qid запросов train/val)
OUT_DIR = f'{WORK}/features'                   # сюда пишем microcat_proba.parquet
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
N_FOLDS = 3          # фолды для out-of-fold предсказаний на запросах CatBoost-train
TOP_N = 30           # сколько микрокатегорий на запрос сохраняем
MIN_CLASS_COUNT = 3  # микрокатегории с меньшим числом примеров не учим (шум)
ALPHA = 5e-6         # регуляризация SGD
MAX_ITER = 8
print('prep:', PREP_DIR, '| features:', FEAT_DIR, '| вывод:', OUT_DIR)

## 1. Данные

- запросы train, val и бенчмарка с текстами (`query_norm`, `query_lem`) и фильтром
- пары «запрос → выбранное объявление» и микрокатегория этого объявления (из **всех** объявлений, не только корпуса)
- `qid` запросов CatBoost-train берём из `features/train.parquet`

In [ ]:
queries = pd.read_parquet(f'{PREP_DIR}/queries.parquet',
                          columns=['qid', 'split', 'query_key', 'query_norm', 'query_lem', 'search_infm_params_text'])
pairs = pd.read_parquet(f'{PREP_DIR}/pairs.parquet', columns=['qid', 'item_id', 'split'])
items = pd.read_parquet(f'{PREP_DIR}/items.parquet', columns=['item_id', 'item_microcat_id', 'title_norm'])
cb_train_qids = set(pd.read_parquet(f'{FEAT_DIR}/train.parquet', columns=['qid'])['qid'].unique())
val_qids = set(pd.read_parquet(f'{FEAT_DIR}/val.parquet', columns=['qid'])['qid'].unique())

mc_of_item = items.set_index('item_id')['item_microcat_id']
pairs['microcat'] = pairs['item_id'].map(mc_of_item)
print(f'запросов: {len(queries):,}, пар: {len(pairs):,}, объявлений: {len(items):,}')
print(f'CatBoost-train запросов: {len(cb_train_qids):,}, val: {len(val_qids):,}, bench: {(queries.split == "bench").sum():,}')

## 2. Текст для классификатора

Три группы признаков, склеенные в одну разреженную матрицу:
- **слова** запроса (леммы, униграммы и биграммы): «ремонт стиральный» → понятно, о чём речь
- **символьные n-граммы** запроса (3–5 символов): ловят опечатки, редкие формы и склейки, которых нет в словаре
- **слова фильтра**: «Вид услуги Красота, здоровье» почти напрямую указывает на микрокатегорию

Векторизаторы обучаются только на текстах train-запросов, без меток. Это не утечка.

In [ ]:
WS_RE = re.compile(r'\s+')
def filter_text(s):
    s = (s or '').lower().replace('ё', 'е')
    return WS_RE.sub(' ', re.sub(r'[^a-zа-я0-9]+', ' ', s)).strip()

queries['q_lem'] = queries['query_lem'].fillna('')
queries['q_chr'] = queries['query_norm'].fillna('').str.lower()
queries['q_flt'] = queries['search_infm_params_text'].fillna('').map(filter_text)

tr_texts = queries[queries.split == 'train']
t = time.time()
vec_word = TfidfVectorizer(token_pattern=r'\S+', ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(tr_texts.q_lem)
vec_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True,
                           max_features=150_000).fit(tr_texts.q_chr)
vec_flt  = TfidfVectorizer(token_pattern=r'\S+', ngram_range=(1, 3), min_df=2, sublinear_tf=True).fit(tr_texts.q_flt)

def vectorize(df):
    return sp.hstack([vec_word.transform(df.q_lem), vec_char.transform(df.q_chr),
                      vec_flt.transform(df.q_flt)]).tocsr()

print(f'словари: слова {len(vec_word.vocabulary_):,}, символы {len(vec_char.vocabulary_):,}, '
      f'фильтр {len(vec_flt.vocabulary_):,} | {time.time() - t:.0f} с')

## 3. Обучающая выборка

Одна строка = (текст запроса, фильтр, микрокатегория выбранного объявления). Одинаковые тексты из разных городов
для классификатора неразличимы, поэтому схлопываем их в одну строку с весом = числу повторов. Это в разы ускоряет обучение.

In [ ]:
tr = (pairs[pairs.split == 'train'].dropna(subset=['microcat'])
      .merge(queries[['qid', 'query_key', 'q_lem', 'q_chr', 'q_flt']], on='qid'))
tr['microcat'] = tr['microcat'].astype('int64')
tr['is_cb'] = tr['qid'].isin(cb_train_qids)

class_counts = tr['microcat'].value_counts()
KEEP_CLASSES = class_counts[class_counts >= MIN_CLASS_COUNT].index
tr = tr[tr.microcat.isin(KEEP_CLASSES)]

def dedup(df):
    """Схлопывает одинаковые (текст, фильтр, микрокатегория), вес = число повторов."""
    return (df.groupby(['q_lem', 'q_chr', 'q_flt', 'microcat'], as_index=False)
              .agg(weight=('qid', 'size'), query_key=('query_key', 'first')))

print(f'пар train с микрокатегорией: {len(tr):,}, классов: {len(KEEP_CLASSES)} '
      f'(отброшено редких: {(class_counts < MIN_CLASS_COUNT).sum()})')
print(f'после схлопывания повторов: {len(dedup(tr)):,} строк')

## 4. Out-of-fold предсказания для запросов CatBoost-train

Тексты запросов CatBoost-train делим на 3 фолда. Для фолда k обучаем классификатор на всех train-парах,
**кроме пар с текстами фолда k**, и предсказываем запросы фолда k. Каждый запрос CatBoost-train предсказан моделью,
которая не видела ни его, ни его «близнецов» из других городов, как и на val.

In [ ]:
def fit_clf(df):
    d = dedup(df)
    clf = SGDClassifier(loss='log_loss', alpha=ALPHA, max_iter=MAX_ITER, tol=1e-3,
                        n_jobs=-1, random_state=SEED)
    t = time.time()
    clf.fit(vectorize(d), d['microcat'].values, sample_weight=d['weight'].values)
    print(f'  обучено на {len(d):,} строк за {time.time() - t:.0f} с')
    return clf

def predict_top(clf, qdf, top_n=TOP_N):
    """-> long-таблица qid, microcat, proba (топ-N микрокатегорий на запрос)."""
    P = clf.predict_proba(vectorize(qdf)).astype('float32')
    top_n = min(top_n, P.shape[1])
    top = np.argsort(-P, axis=1)[:, :top_n]
    return pd.DataFrame({
        'qid': np.repeat(qdf['qid'].values, top_n),
        'microcat': clf.classes_[top].ravel(),
        'proba': np.take_along_axis(P, top, axis=1).ravel(),
    })

rng = np.random.default_rng(SEED)
cb_q = queries[queries.qid.isin(cb_train_qids)]
cb_texts = cb_q['query_key'].unique()
fold_of_text = pd.Series(rng.integers(0, N_FOLDS, len(cb_texts)), index=cb_texts)

oof_parts = []
for k in range(N_FOLDS):
    fold_texts = set(fold_of_text[fold_of_text == k].index)
    print(f'фолд {k}: текстов {len(fold_texts):,}')
    clf_k = fit_clf(tr[~tr.query_key.isin(fold_texts)])
    oof_parts.append(predict_top(clf_k, cb_q[cb_q.query_key.isin(fold_texts)]))
    del clf_k; gc.collect()
oof = pd.concat(oof_parts, ignore_index=True)
print(f'OOF: {oof.qid.nunique():,} запросов CatBoost-train')

## 5. Финальная модель на всех train-запросах → предсказания для val и бенчмарка

In [ ]:
clf_full = fit_clf(tr)
vb_q = queries[queries.qid.isin(val_qids) | (queries.split == 'bench')]
full_pred = predict_top(clf_full, vb_q)
mc_proba = pd.concat([oof, full_pred], ignore_index=True)
print(f'итого: {mc_proba.qid.nunique():,} запросов × топ-{TOP_N} = {len(mc_proba):,} строк')

## 6. Качество классификатора

Для запросов CatBoost-train (OOF) и val: в какой доле случаев микрокатегория выбранного объявления попала в топ-1, топ-3, топ-10
предсказаний. Числа на OOF и val должны быть близки, тогда признак для CatBoost ведёт себя одинаково на обучении и проверке.

In [ ]:
def topk_acc(pred, qids, split_pairs):
    truth = split_pairs[split_pairs.qid.isin(qids)][['qid', 'microcat']].dropna().drop_duplicates()
    ranked = pred[pred.qid.isin(qids)].copy()
    ranked['rank'] = ranked.groupby('qid')['proba'].rank(method='first', ascending=False)
    m = truth.merge(ranked, on=['qid', 'microcat'], how='left')
    best = m.groupby('qid')['rank'].min()             # лучшая позиция среди правильных микрокатегорий запроса
    return {f'top-{k}': float((best <= k).mean()) for k in [1, 3, 10, 30]}

acc = pd.DataFrame({
    'CatBoost-train (OOF)': topk_acc(mc_proba, cb_train_qids, pairs),
    'val': topk_acc(mc_proba, val_qids, pairs),
})
display(acc.style.format('{:.2%}'))

### Как это выглядит: несколько val-запросов и их топ-3 микрокатегории (с примером заголовка из каждой)

In [ ]:
mc_example = (items.dropna(subset=['item_microcat_id']).groupby('item_microcat_id')['title_norm']
              .agg(lambda s: s.value_counts().index[0]))
ex_q = queries[queries.qid.isin(val_qids)].sample(8, random_state=1)
for _, r in ex_q.iterrows():
    top3 = mc_proba[mc_proba.qid == r.qid].nlargest(3, 'proba')
    truth = set(pairs.loc[pairs.qid == r.qid, 'microcat'].dropna().astype('int64'))
    print(f'\n«{r.query_norm}» | фильтр: «{r.search_infm_params_text or ""}»')
    for _, t_ in top3.iterrows():
        mark = '✓' if t_.microcat in truth else ' '
        print(f'   {mark} {t_.proba:.2f}  {t_.microcat}: {str(mc_example.get(t_.microcat, "?"))[:60]}')

## 7. Сохранение

In [ ]:
mc_proba['microcat'] = mc_proba['microcat'].astype('int64')
mc_proba.to_parquet(f'{OUT_DIR}/microcat_proba.parquet', index=False)
print(f'{OUT_DIR}/microcat_proba.parquet: {os.path.getsize(f"{OUT_DIR}/microcat_proba.parquet") / 2**20:.1f} МБ')
display(mc_proba.head(10))